In [2]:
import secrets
from dotenv import load_dotenv, find_dotenv
from doc_chat.rag.document_loader import DocumentLoader
from doc_chat.rag.vector_store import VectorStore
import chromadb

_ = load_dotenv(find_dotenv())


In [5]:
file_path = "../data/pdf/The Hundred-Page Machine Learning Book.pdf"
#file_path = "../data/single_topic_rag_evaluation_dataset/processed/pdf/document_0.pdf"
doc_loader = DocumentLoader(file_path)
documents, splits = doc_loader.load_and_split(
    chunk_size = 1000, chunk_overlap = 0
)
print("Number of documents: " + str(len(documents)))
print("Number of splits: " + str(len(splits)))

Number of documents: 152
Number of splits: 366


In [ ]:
# document format
documents[0]

In [ ]:
# print first 300 characters of the third page
documents[2].page_content[:300]

In [ ]:
# split format
splits[0:2]

In [ ]:
splits[2].page_content

In [ ]:
# instantiate vector store
token = secrets.token_urlsafe(16)
vector_store = VectorStore(documents=splits, token=token)

In [ ]:
# langchain uses ChromaDB running in the same process
client = chromadb.PersistentClient(path=vector_store._persist_directory)
collections = client.list_collections()
print(collections)

# langchain creates a single collection named 'langchain' for the pdf file
# each split is stored as a separate document in the collection
collection = client.get_collection(name="langchain")
print(f"Number of documents in collection '{collection.name}': {collection.count()}")

# print metadata of the collection
print(collection.metadata)

In [ ]:
# get first 3 documents from chroma
results = collection.peek(3)
results

In [ ]:
conversational_chain = vector_store.create_conversational_retrieval_chain(k=2)
response = conversational_chain.invoke("Explain logistic regression")
print(response['answer'])

In [ ]:
source_docs = response['source_documents']
print("Number of source documents: " + str(len(source_docs)))

for i, doc in enumerate(source_docs):
    print(f"Source document {i}")
    print(f"Page: {doc.metadata['page']}")
    print(f"Content: {doc.page_content[:10]}")
    print("-----")

source_docs

In [ ]:
source_docs[0]

In [ ]:
source_docs[1]

In [ ]:
# conversational_chain can be used to ask follow-up questions
response = conversational_chain.invoke("Give me the formula")
print(response['answer'])

In [ ]:
# qa chain does not use the history of the conversation
qa_chain = vector_store.create_qa_chain(k=2)
response = qa_chain.invoke("Explain logistic regression")
print(response['result'])

In [ ]:
# a follow-up question to the qa does not result in answer
response = qa_chain.invoke("Give me the formula")
print(response['result'])

In [ ]:
# retrieve only documents without call to the LLM
documents = vector_store.retrieve_documents("Explain logistic regression", k=2)

for doc in documents:
    print(f"Page: {doc.metadata['page']}")
    print(f"Content: {doc.page_content[:300]}")